# ClimateVision SHAP Explainability

This notebook demonstrates how to use SHAP (SHapley Additive exPlanations) to understand
why the ClimateVision segmentation model makes specific predictions.

**Author:** Linda Oraegbunam (@obielin)  
**Module:** `src/climatevision/governance/explainability.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# ClimateVision imports
from climatevision.governance import explain_prediction, SHAPExplainer, get_band_contributions
from climatevision.inference.pipeline import _load_model
from climatevision.models import UNet

## 1. Understanding SHAP for Segmentation

SHAP values tell us how much each input feature (spectral band) contributed to the model's prediction.
For satellite imagery:
- **Positive SHAP**: Feature pushed prediction toward the target class
- **Negative SHAP**: Feature pushed prediction away from the target class
- **Magnitude**: Strength of the contribution

In [ ]:
# Load the deforestation model
model, device = _load_model('deforestation')
print(f"Model: {model.__class__.__name__}")
print(f"Input channels: {model.n_channels}")
print(f"Output classes: {model.n_classes}")
print(f"Device: {device}")

## 2. Create SHAP Explainer

In [ ]:
# Initialize the explainer with background data
background = torch.zeros(1, model.n_channels, 64, 64).to(device)
explainer = SHAPExplainer(model, background_data=background, device=device)
print("SHAP Explainer initialized")

## 3. Generate Explanation for Sample Image

In [ ]:
# Create a synthetic forest-like image for demonstration
np.random.seed(42)

# Simulate Sentinel-2 bands: Red, Green, Blue, NIR
# Forest typically has high NIR and low Red
h, w = 256, 256
red = np.random.normal(0.2, 0.1, (h, w)).clip(0, 1)  # Low red reflectance
green = np.random.normal(0.3, 0.1, (h, w)).clip(0, 1)
blue = np.random.normal(0.25, 0.1, (h, w)).clip(0, 1)
nir = np.random.normal(0.7, 0.15, (h, w)).clip(0, 1)  # High NIR for vegetation

sample_image = np.stack([red, green, blue, nir], axis=0).astype(np.float32)
sample_tensor = torch.FloatTensor(sample_image).unsqueeze(0).to(device)

print(f"Sample image shape: {sample_image.shape}")

In [ ]:
# Generate SHAP explanation
explanation = explainer.explain(sample_tensor, target_class=1)  # Class 1 = Forest

print("\n=== Explanation Results ===")
print(f"Predicted class: {explanation['prediction']}")
print(f"Target class: {explanation['target_class']}")
print(f"Confidence: {explanation['confidence']:.4f}")
print(f"Explainer type: {explanation['explainer_type']}")
print(f"\nBand contributions:")
for band, importance in explanation['band_contributions'].items():
    print(f"  {band}: {importance:.4f}")

## 4. Visualize Band Contributions

In [ ]:
# Plot band importance
band_names = ['Red (B04)', 'Green (B03)', 'Blue (B02)', 'NIR (B08)']
contributions = explanation['band_contributions']
importances = [contributions[f'band_{i}'] for i in range(len(band_names))]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c', '#27ae60', '#3498db', '#9b59b6']
bars = ax.bar(band_names, importances, color=colors)
ax.set_ylabel('Relative Importance')
ax.set_title('Band Contributions to Forest Classification')
ax.set_ylim(0, max(importances) * 1.2)

# Add value labels
for bar, imp in zip(bars, importances):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{imp:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Spatial Importance Heatmap

In [ ]:
# Visualize spatial importance
spatial_importance = explanation['spatial_importance']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original RGB composite
rgb = np.stack([sample_image[0], sample_image[1], sample_image[2]], axis=-1)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
axes[0].imshow(rgb)
axes[0].set_title('RGB Composite')
axes[0].axis('off')

# SHAP importance heatmap
im = axes[1].imshow(spatial_importance, cmap='hot')
axes[1].set_title('SHAP Importance Heatmap')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

# Overlay
axes[2].imshow(rgb)
axes[2].imshow(spatial_importance, cmap='hot', alpha=0.5)
axes[2].set_title('RGB + SHAP Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 6. Compare Explanations Across Analysis Types

In [ ]:
# Compare band importance across different analysis types
analysis_types = ['deforestation', 'ice_melting', 'flooding']
all_contributions = {}

for atype in analysis_types:
    try:
        model, device = _load_model(atype)
        explainer = SHAPExplainer(model, device=device)
        
        # Create appropriate test tensor
        test_tensor = torch.randn(1, model.n_channels, 128, 128).to(device)
        result = explainer.explain(test_tensor)
        all_contributions[atype] = result['band_contributions']
        print(f"{atype}: {len(result['band_contributions'])} bands analyzed")
    except Exception as e:
        print(f"{atype}: Failed - {e}")

print("\nComparison complete!")

## 7. Using the High-Level API

In [ ]:
# For real usage with saved images:
# result = explain_prediction(
#     model_path='models/unet_deforestation.pth',
#     image_path='data/test/amazon_tile.tif',
#     analysis_type='deforestation',
#     save_heatmap=True
# )
# print(f"Top bands: {result['top_bands']}")
# print(f"Heatmap saved to: {result['heatmap_path']}")

print("See explain_prediction() for file-based explanations")

## Summary

This notebook demonstrated:
1. **SHAPExplainer** - Core class for generating explanations
2. **Band contributions** - Which spectral bands drive predictions
3. **Spatial importance** - Which image regions matter most
4. **Visualization** - Heatmaps and bar charts for stakeholder communication

For production use, call the `/api/explain` endpoint or use `explain_prediction()` directly.